# Credit Risk Analysis - Feature Engineering

This notebook prepares the data for modeling by:
- Dropping useless and leakage features
- Creating binary target variable
- Adding temporal features
- Encoding categorical variables
- Handling missing values
- Scaling features
- Creating train/test split

## 1. Setup and Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/sample_100k_stratified.csv', low_memory=False)
print(f"Data loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Data loaded: 100,000 rows x 152 columns


## 2. Create Binary Target Variable

In [2]:
default_statuses = ['Charged Off', 'Default', 'Late (31-120 days)', 'Late (16-30 days)']

df['is_default'] = df['loan_status'].apply(lambda x: 1 if x in default_statuses else 0)

print(f"Default rate: {df['is_default'].mean()*100:.2f}%")
print(f"Defaults: {df['is_default'].sum():,}")
print(f"Non-defaults: {(len(df) - df['is_default'].sum()):,}")

Default rate: 12.84%
Defaults: 12,842
Non-defaults: 87,158


## 3. Drop High-Missing Features (>95% missing)

In [4]:
# Calculate missing percentages
missing_pct = (df.isnull().sum() / len(df)) * 100
high_missing = missing_pct[missing_pct > 95].sort_values(ascending=False)

print(f"Features with >95% missing: {len(high_missing)}")
print("\nDropping these features:")
print(high_missing.head(10))

# Drop high-missing features
df = df.drop(columns=high_missing.index.tolist())
print(f"\nDataset shape after dropping: {df.shape}")

Features with >95% missing: 0

Dropping these features:
Series([], dtype: float64)

Dataset shape after dropping: (100000, 119)


## 4. Drop Data Leakage Features

### Important Note on Data Leakage

**From our EDA, the "top risk factors" were:**
- `recoveries` (0.49 correlation)
- `collection_recovery_fee` (0.46 correlation)
- `total_rec_late_fee` (0.13 correlation)

**We cannot use these because:**

These features are **consequences of default, not predictors!** They represent events that happen **AFTER** a loan has already defaulted:

1. **Loan defaults** → borrower stops paying
2. **Collections begin** → lender tries to recover money
3. **Recoveries & fees** → these values get recorded

If we included these features, our model would have perfect hindsight:
- "If `recoveries` > 0, then the loan defaulted" 
- This is data leakage (using future information to predict the past)

**Features to drop (post-default data):**
- `recoveries`
- `collection_recovery_fee`
- `total_rec_late_fee`
- `last_pymnt_amnt`
- `last_pymnt_d`
- `total_rec_prncp`
- `total_rec_int`
- `total_pymnt`
- `total_pymnt_inv`
- `out_prncp`
- `out_prncp_inv`
- `last_fico_range_high`
- `last_fico_range_low`
- `last_credit_pull_d`

**What we will use (pre-loan or at-origination data):**
- `fico_range_low/high` (at origination)
- `int_rate` (assigned at origination)
- `grade` (assigned at origination)
- `annual_inc`, `dti`, `emp_length` (at application time)

In [5]:
# List of leakage features to drop
leakage_features = [
    'recoveries', 'collection_recovery_fee', 'total_rec_late_fee',
    'last_pymnt_amnt', 'last_pymnt_d', 'total_rec_prncp', 'total_rec_int',
    'total_pymnt', 'total_pymnt_inv', 'out_prncp', 'out_prncp_inv',
    'last_fico_range_high', 'last_fico_range_low', 'last_credit_pull_d',
    'next_pymnt_d', 'funded_amnt', 'funded_amnt_inv', 'installment',
    'total_pymnt', 'issue_d'  # Keep issue_date separately for temporal features
]

# Drop leakage features that exist in the dataframe
leakage_to_drop = [f for f in leakage_features if f in df.columns]
print(f"Dropping {len(leakage_to_drop)} data leakage features...")

df = df.drop(columns=leakage_to_drop)
print(f"Dataset shape after removing leakage: {df.shape}")

Dropping 20 data leakage features...
Dataset shape after removing leakage: (100000, 100)


## 5. Add Temporal Features

In [6]:
# Extract temporal features from issue_d (before we dropped it)
# Reload to get issue_d
df_temp = pd.read_csv('../data/sample_100k_stratified.csv', usecols=['issue_d'], low_memory=False)
df['issue_d'] = df_temp['issue_d']

# Create temporal features
df['issue_date'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['issue_year'] = df['issue_date'].dt.year
df['issue_month'] = df['issue_date'].dt.month
df['is_post_2015'] = (df['issue_year'] >= 2015).astype(int)

# Drop the date columns (keep numeric features only)
df = df.drop(columns=['issue_d', 'issue_date'])

print("Temporal features added:")
print("   - issue_year")
print("   - issue_month")
print("   - is_post_2015 (binary)")

Temporal features added:
   - issue_year
   - issue_month
   - is_post_2015 (binary)


## 6. Drop Non-Predictive Features

In [7]:
# Drop ID columns and non-predictive features
non_predictive = ['id', 'member_id', 'url', 'desc', 'title', 'zip_code', 
                 'emp_title', 'loan_status', 'pymnt_plan', 'policy_code']

to_drop = [f for f in non_predictive if f in df.columns]
df = df.drop(columns=to_drop)

print(f"Dropped {len(to_drop)} non-predictive features")
print(f"   Dataset shape: {df.shape}")

Dropped 9 non-predictive features
   Dataset shape: (100000, 93)


## 7. Feature Selection - Keep Important Features

In [8]:
# Select key features for modeling
# Based on EDA insights and domain knowledge

key_features = [
    # Target
    'is_default',
    
    # Temporal
    'issue_year', 'issue_month', 'is_post_2015',
    
    # Loan characteristics (at origination)
    'loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade',
    
    # Borrower characteristics
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    
    # Credit history (at origination)
    'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
    'total_acc', 'earliest_cr_line',
    
    # Loan purpose
    'purpose',
    
    # Application type
    'application_type'
]

# Keep only features that exist in the dataframe
available_features = [f for f in key_features if f in df.columns]
df_model = df[available_features].copy()

print(f"Selected {len(available_features)} key features for modeling")
print(f"   Dataset shape: {df_model.shape}")
print(f"\nFeatures: {available_features}")

Selected 26 key features for modeling
   Dataset shape: (100000, 26)

Features: ['is_default', 'issue_year', 'issue_month', 'is_post_2015', 'loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'earliest_cr_line', 'purpose', 'application_type']


## 8. Handle Missing Values

In [9]:
# Check missing values in selected features
missing_summary = df_model.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)

print("Missing values in selected features:")
print(missing_summary)
print(f"\nTotal features with missing values: {len(missing_summary)}")

Missing values in selected features:
emp_length             6505
revol_util               99
dti                      84
delinq_2yrs               8
earliest_cr_line          8
total_acc                 8
pub_rec                   8
open_acc                  8
inq_last_6mths            8
fico_range_low            1
purpose                   1
revol_bal                 1
fico_range_high           1
issue_year                1
issue_month               1
verification_status       1
annual_inc                1
home_ownership            1
sub_grade                 1
grade                     1
int_rate                  1
term                      1
loan_amnt                 1
application_type          1
dtype: int64

Total features with missing values: 24


In [10]:
# Strategy for handling missing values

# For numerical features: fill with median
numerical_cols = df_model.select_dtypes(include=['float64', 'int64']).columns
numerical_cols = [c for c in numerical_cols if c != 'is_default']

for col in numerical_cols:
    if df_model[col].isnull().sum() > 0:
        df_model[col].fillna(df_model[col].median(), inplace=True)

# For categorical features: fill with mode or 'Unknown'
categorical_cols = df_model.select_dtypes(include=['object']).columns

for col in categorical_cols:
    if df_model[col].isnull().sum() > 0:
        df_model[col].fillna('Unknown', inplace=True)

print("Missing values handled")
print(f"   Remaining missing values: {df_model.isnull().sum().sum()}")

Missing values handled
   Remaining missing values: 0


## 9. Encode Categorical Variables

In [11]:
# Get categorical columns
categorical_cols = df_model.select_dtypes(include=['object']).columns.tolist()

print(f"Categorical features to encode: {len(categorical_cols)}")
print(categorical_cols)

# One-hot encode categorical variables
df_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

print(f"\nEncoding complete")
print(f"   Dataset shape after encoding: {df_encoded.shape}")

Categorical features to encode: 9
['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'verification_status', 'earliest_cr_line', 'purpose', 'application_type']

Encoding complete
   Dataset shape after encoding: (100000, 740)


## 10. Prepare Features and Target

In [12]:
# Separate features and target
X = df_encoded.drop('is_default', axis=1)
y = df_encoded['is_default']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nDefault rate in full dataset: {y.mean()*100:.2f}%")

Features (X): (100000, 739)
Target (y): (100000,)

Default rate in full dataset: 12.84%


## 11. Train/Test Split

In [13]:
# Split into train and test sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nDefault rate in train: {y_train.mean()*100:.2f}%")
print(f"Default rate in test: {y_test.mean()*100:.2f}%")

Training set: (80000, 739)
Test set: (20000, 739)

Default rate in train: 12.84%
Default rate in test: 12.84%


**Note on equal default rates across train/test sets:**

The default rate is identical in both the training set (12.84%) and test set (12.84%) because we use `stratify=y` in `train_test_split`. This is an intentional design choice. Stratified splitting preserves the original class proportions in each subset, ensuring that:

1. The test set is representative of the full dataset and not accidentally enriched or depleted of defaults by random chance.
2. Model evaluation metrics (e.g. precision, recall, AUC) are computed against a distribution that reflects real-world conditions.
3. Results are reproducible and not sensitive to a lucky/unlucky random split.

This is especially important given the class imbalance (~87% non-default vs ~13% default). Without stratification, a single bad split could noticeably skew the default rate in either subset.

## 12. Feature Scaling

In [14]:
# Scale features using StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames to preserve column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("Features scaled using StandardScaler")
print(f"   Train: {X_train_scaled.shape}")
print(f"   Test: {X_test_scaled.shape}")

Features scaled using StandardScaler
   Train: (80000, 739)
   Test: (20000, 739)


## 13. Save Processed Data

In [15]:
# Save processed datasets
import pickle

# Save train/test splits
X_train_scaled.to_csv('../data/X_train.csv', index=False)
X_test_scaled.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

# Save scaler for future use
with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Processed data saved:")
print("   - data/X_train.csv")
print("   - data/X_test.csv")
print("   - data/y_train.csv")
print("   - data/y_test.csv")
print("   - data/scaler.pkl")

Processed data saved:
   - data/X_train.csv
   - data/X_test.csv
   - data/y_train.csv
   - data/y_test.csv
   - data/scaler.pkl


## 14. Summary

In [ ]:
print("="*70)
print("FEATURE ENGINEERING SUMMARY")
print("="*70)

print(f"\n1. DATA PIPELINE:")
print(f"   Original features: 154")
print(f"   After dropping high-missing: ~121")
print(f"   After dropping leakage: ~90")
print(f"   After feature selection: {len(available_features)}")
print(f"   After encoding: {X.shape[1]} features")

print(f"\n2. TRAIN/TEST SPLIT:")
print(f"   Training samples: {len(X_train):,} (80%)")
print(f"   Test samples: {len(X_test):,} (20%)")
print(f"   Features: {X_train.shape[1]}")

print(f"\n3. CLASS BALANCE:")
print(f"   Train - Defaults: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"   Train - Non-defaults: {len(y_train)-y_train.sum():,} ({(1-y_train.mean())*100:.2f}%)")
print(f"   Test  - Defaults: {y_test.sum():,} ({y_test.mean()*100:.2f}%)")
print(f"   Test  - Non-defaults: {len(y_test)-y_test.sum():,} ({(1-y_test.mean())*100:.2f}%)")

print(f"\n4. KEY FEATURES INCLUDED:")
print(f"   ✓ Temporal: issue_year, issue_month, is_post_2015")
print(f"   ✓ FICO scores (at origination)")
print(f"   ✓ Loan grade & interest rate")
print(f"   ✓ Borrower income, DTI, employment")
print(f"   ✓ Credit history features")

print(f"\n5. DROPPED FEATURES:")
print(f"   ✗ High-missing (>95%): ~33 features")
print(f"   ✗ Data leakage (post-default): ~15 features")
print(f"   ✗ Non-predictive (IDs, URLs): ~10 features")


FEATURE ENGINEERING SUMMARY

1. DATA PIPELINE:
   Original features: 154
   After dropping high-missing: ~121
   After dropping leakage: ~90
   After feature selection: 26
   After encoding: 739 features

2. TRAIN/TEST SPLIT:
   Training samples: 80,000 (80%)
   Test samples: 20,000 (20%)
   Features: 739

3. CLASS BALANCE:
   Train - Defaults: 10,274 (12.84%)
   Train - Non-defaults: 69,726 (87.16%)
   Test  - Defaults: 2,568 (12.84%)
   Test  - Non-defaults: 17,432 (87.16%)

4. KEY FEATURES INCLUDED:
   ✓ Temporal: issue_year, issue_month, is_post_2015
   ✓ FICO scores (at origination)
   ✓ Loan grade & interest rate
   ✓ Borrower income, DTI, employment
   ✓ Credit history features

5. DROPPED FEATURES:
   ✗ High-missing (>95%): ~33 features
   ✗ Data leakage (post-default): ~15 features
   ✗ Non-predictive (IDs, URLs): ~10 features

Data is ready for modeling.


## 15. Addressing Red Flags

### Red Flag 1: Feature Explosion (26 --> 739 features)

One-hot encoding caused an explosion in the number of features:
- `sub_grade`: 35 categories (A1-G5) → 34 dummy columns. 
- `earliest_cr_line`: Date strings with hundreds of unique values (e.g., 'Jan-1990'). Should be converted to numeric (years of credit history)
- `emp_length`: String categories like '10+ years'. Should be converted to numeric
- `term`: String '36 months'. Should be converted to numeric

**We are concerned because too many features can cause overfitting and slow training**

### Red Flag 2: Class Imbalance (12.84% defaults)

- 87% non-default vs 13% default
- A naive model predicting 'non-default' for everything would score 87% accuracy
- **Solution:** Use `class_weight='balanced'` in models and evaluate with precision, recall, F1, and ROC-AUC

### Red Flag 3: Redundant Features

- `fico_range_low` and `fico_range_high` are highly correlated (differ by ~4 points)
- `sub_grade` is just a finer version of `grade`
- Keeping both adds noise without new information

**Below we address all these issues.**

In [21]:
# FIX 1: Convert earliest_cr_line to numeric (years of credit history) 
df_temp2 = pd.read_csv('../data/sample_100k_stratified.csv', usecols=['issue_d'], low_memory=False)
issue_dates = pd.to_datetime(df_temp2['issue_d'], format='%b-%Y')

cr_line_dates = pd.to_datetime(df_model['earliest_cr_line'], format='%b-%Y', errors='coerce')
df_model['credit_history_years'] = (issue_dates - cr_line_dates).dt.days / 365.25
df_model = df_model.drop(columns=['earliest_cr_line'])

print('Fix 1: Converted earliest_cr_line -> credit_history_years')
print(f'   Median credit history: {df_model["credit_history_years"].median():.1f} years')

Fix 1: Converted earliest_cr_line -> credit_history_years
   Median credit history: 14.8 years


In [22]:
# FIX 2: Drop sub_grade (redundant with grade)
df_model = df_model.drop(columns=['sub_grade'])
print('Fix 2: Dropped sub_grade (redundant with grade)')

Fix 2: Dropped sub_grade (redundant with grade)


In [23]:
# FIX 3: Convert term to numeric 
df_model['term'] = df_model['term'].str.extract('(\d+)').astype(float)
print('Fix 3: Converted term to numeric')
print(f'   Unique values: {sorted(df_model["term"].dropna().unique())}')

Fix 3: Converted term to numeric
   Unique values: [np.float64(36.0), np.float64(60.0)]


In [24]:
# FIX 4: Convert emp_length to numeric 
emp_mapping = {
    '< 1 year': 0.5, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8, '9 years': 9, '10+ years': 10, 'Unknown': np.nan
}
df_model['emp_length'] = df_model['emp_length'].map(emp_mapping)
df_model['emp_length'].fillna(df_model['emp_length'].median(), inplace=True)

print('Fix 4: Converted emp_length to numeric')
print(f'   Median employment length: {df_model["emp_length"].median():.1f} years')

Fix 4: Converted emp_length to numeric
   Median employment length: 6.0 years


In [25]:
# FIX 5: Combine FICO range into single feature 
df_model['fico_score'] = (df_model['fico_range_low'] + df_model['fico_range_high']) / 2
df_model = df_model.drop(columns=['fico_range_low', 'fico_range_high'])

print('Fix 5: Combined fico_range_low and fico_range_high into fico_score')
print(f'   Median FICO score: {df_model["fico_score"].median():.0f}')

Fix 5: Combined fico_range_low and fico_range_high into fico_score
   Median FICO score: 692


In [26]:
# Check remaining categorical features 
remaining_cats = df_model.select_dtypes(include=['object']).columns.tolist()
print(f'Remaining categorical features: {len(remaining_cats)}')
for col in remaining_cats:
    print(f'   {col}: {df_model[col].nunique()} unique values')

Remaining categorical features: 5
   grade: 8 unique values
   home_ownership: 7 unique values
   verification_status: 4 unique values
   purpose: 15 unique values
   application_type: 3 unique values


## 16. Re-encode, Split, and Scale (Fixed)

In [27]:
remaining_cats = df_model.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df_model, columns=remaining_cats, drop_first=True)

print(f'Dataset shape after encoding: {df_encoded.shape}')
print(f'\nFeature count BEFORE fixes: 739')
print(f'Feature count AFTER fixes:  {df_encoded.shape[1] - 1}')

Dataset shape after encoding: (100000, 51)

Feature count BEFORE fixes: 739
Feature count AFTER fixes:  50


In [28]:
# Separate features and target
X = df_encoded.drop('is_default', axis=1)
y = df_encoded['is_default']

# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f'Training set: {X_train_scaled.shape}')
print(f'Test set: {X_test_scaled.shape}')
print(f'\nDefault rate in train: {y_train.mean()*100:.2f}%')
print(f'Default rate in test: {y_test.mean()*100:.2f}%')

Training set: (80000, 50)
Test set: (20000, 50)

Default rate in train: 12.84%
Default rate in test: 12.84%


## 17. Save Fixed Data

In [29]:
import pickle

# Save the fixed train/test splits
X_train_scaled.to_csv('../data/X_train.csv', index=False)
X_test_scaled.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature names for modeling
with open('../data/feature_names.pkl', 'wb') as f:
    pickle.dump(X_train.columns.tolist(), f)

print('Processed data saved (overwritten with fixed version)')

Processed data saved (overwritten with fixed version)


## 18. Final Summary

In [30]:
print('='*70)
print('REVISED FEATURE ENGINEERING SUMMARY')
print('='*70)

print(f'\n1. FEATURE REDUCTION:')
print(f'   Original features: 154')
print(f'   After cleaning: 26 selected features')
print(f'   After encoding (BEFORE fixes): 739 features')
print(f'   After encoding (AFTER fixes):  {X.shape[1]} features')
print(f'   Reduction: {739 - X.shape[1]} features removed ({(739 - X.shape[1])/739*100:.0f}% reduction)')

print(f'\n2. FIXES APPLIED:')
print(f'   Fix 1: earliest_cr_line -> credit_history_years (numeric)')
print(f'   Fix 2: Dropped sub_grade (redundant with grade)')
print(f'   Fix 3: term -> numeric (36 or 60)')
print(f'   Fix 4: emp_length -> numeric (0.5 to 10)')
print(f'   Fix 5: fico_range_low + fico_range_high -> fico_score')

print(f'\n3. TRAIN/TEST SPLIT:')
print(f'   Training: {len(X_train):,} samples ({X_train.shape[1]} features)')
print(f'   Test: {len(X_test):,} samples ({X_test.shape[1]} features)')

print(f'\n4. CLASS IMBALANCE (to address in modeling):')
print(f'   Default rate: {y.mean()*100:.2f}%')
print(f'   -> Use class_weight="balanced" in models')
print(f'   -> Evaluate with precision, recall, F1, ROC-AUC (NOT accuracy)')

print(f'\n5. FINAL FEATURES ({X.shape[1]}):')
for col in X.columns:
    print(f'   - {col}')


REVISED FEATURE ENGINEERING SUMMARY

1. FEATURE REDUCTION:
   Original features: 154
   After cleaning: 26 selected features
   After encoding (BEFORE fixes): 739 features
   After encoding (AFTER fixes):  50 features
   Reduction: 689 features removed (93% reduction)

2. FIXES APPLIED:
   Fix 1: earliest_cr_line -> credit_history_years (numeric)
   Fix 2: Dropped sub_grade (redundant with grade)
   Fix 3: term -> numeric (36 or 60)
   Fix 4: emp_length -> numeric (0.5 to 10)
   Fix 5: fico_range_low + fico_range_high -> fico_score

3. TRAIN/TEST SPLIT:
   Training: 80,000 samples (50 features)
   Test: 20,000 samples (50 features)

4. CLASS IMBALANCE (to address in modeling):
   Default rate: 12.84%
   -> Use class_weight="balanced" in models
   -> Evaluate with precision, recall, F1, ROC-AUC (NOT accuracy)

5. FINAL FEATURES (50):
   - issue_year
   - issue_month
   - is_post_2015
   - loan_amnt
   - term
   - int_rate
   - emp_length
   - annual_inc
   - dti
   - delinq_2yrs
   - in